# deepchunk_run — E5: chấm sâu vùng ranh giới top-10

**Một lượt GPU, ba kết quả.** Ước **1h15m – 1h45m**. Bật GPU T4, **Save & Run All (Commit)**.

Mục tiêu: 17 câu trên dev300 có gold nằm hạng 6-10 của cross-encoder, mà cả 17 văn bản
đó đều dài hơn 3 đoạn (trung vị 125, max 744). D chỉ đưa 3 đoạn/văn bản nên CE đang
chấm văn bản đúng bằng bằng chứng sai. Trần lý thuyết **+5,4 điểm**, kỳ vọng thật +2..+3.

Notebook này sinh ra:

1. `scores_dev300_AITeamVN_Vietnamese_Reranker.json` — **file scores của model GỐC đang
   thiếu trên máy**. Lưu ngay sau tầng 1, trước khi làm gì thêm. Kể cả tầng 2 hỏng thì
   lượt GPU này vẫn lãi.
2. `scores_dev300_deep_M10_K20.json` — có thêm khoá `ce_deep`, `ce` giữ nguyên.
3. `deepchunk_eval.json` — bảng 3 biến thể × 4 giá trị `n_bm25`.

> **KHÔNG dùng model fine-tune ở đây.** Chạy model gốc để chỉ đổi MỘT biến (cách chọn
> đoạn). Fine-tune đã đo là vô hiệu — xem CLAUDE.md mục E3.

## Cài package + trỏ dataset

In [ ]:
!pip install -q sentence-transformers

import os, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
INPUT_DIR = "/kaggle/input/project-ir"     # đổi đúng slug — xem output bên dưới
sys.path.append(INPUT_DIR)
!ls /kaggle/input

In [ ]:
import json, time
from pathlib import Path

from metrics import evaluate
from rerank import load_reranker
from rerank_from_d import score_all_from_d, blend_bm25_first
import deep_chunk as DC

## Config

In [ ]:
RERANKER_MODEL = "AITeamVN/Vietnamese_Reranker"   # GỐC, không phải ft_model
DEVICE = "cuda"

DEV_GOLD  = f"{INPUT_DIR}/dev_300_locked.json"
DEV_CAND  = f"{INPUT_DIR}/bm25_top100_dev300.json"
CTX_DIR   = f"{INPUT_DIR}/selected-contexts"

M_DOC   = 10    # chấm sâu bao nhiêu VĂN BẢN đầu mỗi câu. >5 vì recall@5 là tập hợp
K_CHUNK = 20    # bao nhiêu ĐOẠN mỗi văn bản, thay cho 3 đoạn của D
TOPK    = 5

BASELINE = 0.8883        # AITeamVN gốc + blend n=2, dev300
OUTPUT_DIR = "/kaggle/working/outputs"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## Bước 1 — Đọc dev + candidate

Kiểm `selected-contexts` có mặt: tầng 2 cần toàn văn, tầng 1 thì không.

In [ ]:
dev  = json.load(open(DEV_GOLD, encoding="utf-8"))
cand = json.load(open(DEV_CAND, encoding="utf-8"))
dev_q    = {q: v["question"] for q, v in dev.items()}
dev_gold = {q: v["answer"]   for q, v in dev.items()}

thieu = set(dev_q) - set(cand)
assert not thieu, f"{len(thieu)} câu thiếu candidate"
assert os.path.isdir(CTX_DIR), f"KHÔNG thấy {CTX_DIR} — tầng 2 cần toàn văn corpus"

n1 = sum(len(c.get("top_chunks") or []) for q in cand for c in cand[q])
print(f"{len(dev)} câu | tầng 1: {n1:,} đoạn ({n1/len(dev):.0f}/câu)")
print(f"corpus: {len(os.listdir(CTX_DIR)):,} file")

## Bước 2 — Load reranker

In [ ]:
score_fn = load_reranker(RERANKER_MODEL, device=DEVICE)

## Bước 3 — TẦNG 1: chấm như cũ, rồi LƯU NGAY

Đây là mốc 0.8883 được dựng lại. **Lưu trước khi chạy tầng 2** — quy tắc 2 trong
CLAUDE.md: mỗi lượt GPU phải sinh scores và phải giữ được, dù phần sau có hỏng.

In [ ]:
tag = RERANKER_MODEL.replace("/", "_")
t0 = time.time()
scores_base = score_all_from_d(dev_q, cand, score_fn)
print(f"tầng 1 xong trong {(time.time()-t0)/60:.0f} phút")

p1 = f"{OUTPUT_DIR}/scores_dev300_{tag}.json"
json.dump(scores_base, open(p1, "w", encoding="utf-8"), ensure_ascii=False)
print(f"ĐÃ LƯU {p1} — TẢI FILE NÀY VỀ dù phần sau có hỏng")

## Bước 4 — TẦNG 2: chấm sâu M văn bản đầu

Đếm số đoạn trước (CPU, vài giây) rồi mới chấm. Nếu con số vượt xa ~60.000 thì
dừng lại xem `M_DOC`/`K_CHUNK` có bị đặt sai không, đừng để nó chạy mù.

In [ ]:
t0 = time.time()
n2 = DC.count_deep_chunks(dev_q, scores_base, CTX_DIR, M_DOC, K_CHUNK)
print(f"tầng 2 sẽ chấm {n2:,} đoạn ({n2/len(dev_q):.0f}/câu) | băm+đếm {time.time()-t0:.0f}s")
print(f"tổng {n1+n2:,} đoạn = {(n1+n2)/n1:.2f}x lượt dev300 thường")
assert n2 < 150_000, "quá nhiều — kiểm lại M_DOC / K_CHUNK trước khi đốt GPU"

t0 = time.time()
scores_deep = DC.deepen_all(dev_q, scores_base, CTX_DIR, score_fn, M_DOC, K_CHUNK)
print(f"tầng 2 xong trong {(time.time()-t0)/60:.0f} phút")

p2 = f"{OUTPUT_DIR}/scores_dev300_deep_M{M_DOC}_K{K_CHUNK}.json"
json.dump(scores_deep, open(p2, "w", encoding="utf-8"), ensure_ascii=False)
print(f"ĐÃ LƯU {p2}")

## Bước 5 — Đo: 3 biến thể × 4 giá trị `n_bm25`

`base` phải ra **đúng 0.8883** ở `n=2`. Nếu lệch thì thiết lập sai ở đâu đó —
dừng lại, đừng đọc tiếp mấy dòng dưới.

`n_bm25` **bắt buộc quét lại**: nó được chốt cho bảng điểm cũ. Điểm đổi thì đỉnh
có thể dời. Đây đúng là lỗi đã dính hôm 11/08 khi chốt n=1 trên dev150.

In [ ]:
bm25 = {q: [str(c["doc_id"]) for c in cand[q]] for q in dev_q}

def rec(S, variant, n):
    pred = {q: blend_bm25_first(DC.rank_by(S[q], variant), bm25[q], k=TOPK, n_bm25=n)
            for q in dev_q}
    return evaluate(dev_gold, pred, k=TOPK)["recall"]

tab = {v: {n: rec(scores_deep, v, n) for n in (0, 1, 2, 3)} for v in DC.VARIANTS}

print(f"{'biến thể':10s}" + "".join(f"  n={n}    " for n in (0,1,2,3)))
for v, row in tab.items():
    print(f"{v:10s}" + "".join(f"  {row[n]:.4f} " for n in (0,1,2,3)))

chk = tab["base"][2]
print(f"\nkiểm chứng: base n=2 = {chk:.4f} (phải là {BASELINE})"
      + ("  OK" if abs(chk-BASELINE) < 0.002 else "  <-- LỆCH, dừng lại xem lại thiết lập"))

best_v, best_n = max(((v,n) for v in tab for n in tab[v]), key=lambda x: tab[x[0]][x[1]])
best = tab[best_v][best_n]
print(f"\ntốt nhất: {best_v} n_bm25={best_n} -> {best:.4f}  (Δ {best-BASELINE:+.4f})")
print("=> " + ("ĐÁNG DÙNG. Chạy finalAnswer_run với cấu hình này."
               if best - BASELINE >= 0.02 else
               "CHƯA ĐỦ. Cần ≥ +2,0 điểm mới vượt nhiễu của dev300 (quy tắc 4). "
               "Kết luận: chọn đoạn theo từ trùng đã hết đường -> giục D làm bi-encoder D4."))

## Bước 6 — Lưu + danh sách file phải tải về

Kaggle xoá `/kaggle/working` sau phiên. **Tải hết `outputs/` về trước khi đóng.**

In [ ]:
json.dump({"table": tab, "baseline": BASELINE, "best": [best_v, best_n, best],
           "delta": best - BASELINE, "model": RERANKER_MODEL,
           "M_DOC": M_DOC, "K_CHUNK": K_CHUNK,
           "n_chunk_tang1": n1, "n_chunk_tang2": n2},
          open(f"{OUTPUT_DIR}/deepchunk_eval.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)

for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f}  {os.path.getsize(os.path.join(OUTPUT_DIR,f)):,} bytes")
print("\nTẢI VỀ TOÀN BỘ outputs/ — nhất là scores_dev300_*.json")